In [ ]:
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# all figures are written here
OUT_DIR = "/data/outputs/report"
os.makedirs(OUT_DIR, exist_ok=True)


# Best run per method vs mTAND variant

Comparing the best m3care and fusemoe runs against their mTAND counterparts
(all on the 3.1_ohe bundle, `deterioration_mechanical_ventilation`).
The mTAND runs use the `allRecord` bundle and only ship test-split bootstrap
metrics, so the comparison is test-set only.

In [ ]:
m3care_path = "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe"
fusemoe_path = "/data/logs/ablation/smp/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe"
mtand_m3care_path = "/data/logs/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/mtand_m3care_similarityloss_w0_01_imputedMiss_maskFusion_tm8_et16_3_1ohe"
mtand_fusemoe_path = "/data/logs/ablation/fusemoe/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/fusemoe_fusemoe_0_333SharedWeight_0_1ModalityDropout_0_1EmbedDropout_1e_1Lb_mtandOn_3_1ohe"

# display name -> run directory; order fixes the bar order in every figure
model_paths = {
    "m3care": m3care_path,
    "m3care + mTAND": mtand_m3care_path,
    "fusemoe": fusemoe_path,
    "fusemoe + mTAND": mtand_fusemoe_path,
}

# MM-SHAP modality contribution matrix

Global MM-SHAP modality shares for the best run of each fusion method, across every
outcome of both pathways — one stacked bar per run, so agreement between methods is
visible as three near-identical bars in a group and disagreement as divergent ones.
Shares are global (missingness-diluted): a CXR exists for only ~22% of deterioration
test stays, so the cxr segment understates its weight in the subgroup that has one. The
companion dot-plot below therefore shows each sparse modality's share within the
subgroup that has it (availability inferred from a non-zero per-sample attribution, and
global share = availability x conditional share, since an absent modality attributes 0).

The FuseMoE + mTAND entries here and in the fairness headline reuse
`mtand_fusemoe_path` from the performance comparison above.


In [ ]:
import json

DET_ABL = "/data/logs/ablation/{m}/deterioration/allStay/averageRecord/allCohort/normalised_ohe/{o}/runs/{r}"
DET_MTAND = "/data/logs/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/{r}"
HOSP_ABL = "/data/logs/ablation/{m}/hospitalisation/{w}/averageRecord/optionalHistory/normalised_ohe/{o}/runs/{r}"

mm_shap_runs = {
    "deterioration": [
        ("Cardiac arrest", "concat", DET_ABL.format(m="concat", o="deterioration_cardiac_arrest", r="concat_concat_lr5e5_wd1e3_s42_ohe_cardiac_arrest")),
        ("Cardiac arrest", "m3care", DET_ABL.format(m="m3care", o="deterioration_cardiac_arrest", r="m3care_m3care_similarityloss_w0_01_imputedMiss_maskFusion_cardiac_arrest_3_1ohe")),
        ("Cardiac arrest", "fusemoe", DET_ABL.format(m="smp", o="deterioration_cardiac_arrest", r="smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe_deterioration_cardiac_arrest")),
        ("ICU within 24h", "concat", DET_ABL.format(m="concat", o="deterioration_icu_24h", r="concat_concat_lr5e5_wd1e3_s42_ohe_icu_24h")),
        ("ICU within 24h", "m3care", DET_ABL.format(m="m3care", o="deterioration_icu_24h", r="m3care_m3care_similarityloss_w0_01_imputedMiss_maskFusion_icu_24h_3_1ohe")),
        ("ICU within 24h", "fusemoe", DET_ABL.format(m="smp", o="deterioration_icu_24h", r="smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe_deterioration_icu_24h")),
        ("ICU admission", "concat", DET_ABL.format(m="concat", o="deterioration_icu_stay", r="concat_concat_lr5e5_wd1e3_s42_ohe_icu_stay")),
        ("ICU admission", "m3care", DET_ABL.format(m="m3care", o="deterioration_icu_stay", r="m3care_m3care_similarityloss_w0_01_imputedMiss_maskFusion_icu_stay_3_1ohe")),
        ("ICU admission", "fusemoe", DET_ABL.format(m="smp", o="deterioration_icu_stay", r="smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe_deterioration_icu_stay")),
        ("One-day mortality", "concat", DET_ABL.format(m="concat", o="deterioration_mortality_1d", r="concat_concat_lr5e5_wd1e3_s42_ohe_mortality_1d")),
        ("One-day mortality", "m3care", DET_ABL.format(m="m3care", o="deterioration_mortality_1d", r="m3care_m3care_similarityloss_w0_01_imputedMiss_maskFusion_mortality_1d_3_1ohe")),
        ("One-day mortality", "fusemoe", DET_ABL.format(m="smp", o="deterioration_mortality_1d", r="smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe_deterioration_mortality_1d")),
        ("Mechanical ventilation", "concat", DET_ABL.format(m="concat", o="deterioration_mechanical_ventilation", r="concat_concat_lr5e5_wd1e3_s42_ohe")),
        ("Mechanical ventilation", "m3care", DET_ABL.format(m="m3care", o="deterioration_mechanical_ventilation", r="m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe")),
        ("Mechanical ventilation", "fusemoe", DET_ABL.format(m="smp", o="deterioration_mechanical_ventilation", r="smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe")),
        ("Mech. ventilation (mTAND)", "m3care + mTAND", DET_MTAND.format(r="mtand_m3care_similarityloss_w0_01_imputedMiss_maskFusion_tm8_et16_3_1ohe")),
        ("Mech. ventilation (mTAND)", "fusemoe + mTAND", mtand_fusemoe_path),
    ],
    "hospitalisation": [
        ("Ext. stay > 7d (dual)", "concat", HOSP_ABL.format(m="concat", w="dualWindow", o="hospitalisation_ext_stay_7", r="concat_concat_lr1e5_wd1e4_s42")),
        ("Ext. stay > 7d (dual)", "m3care", HOSP_ABL.format(m="m3care", w="dualWindow", o="hospitalisation_ext_stay_7", r="m3care_m3care_imputedMiss_maskFusion_3_1hosp")),
        ("Ext. stay > 7d (dual)", "fusemoe", HOSP_ABL.format(m="smp", w="dualWindow", o="hospitalisation_ext_stay_7", r="smp_smp_1e_1Lb_3_1hosp")),
        ("Non-home disch. (dual)", "concat", HOSP_ABL.format(m="concat", w="dualWindow", o="hospitalisation_non_home_discharge", r="concat_concat_lr1e5_wd1e4_s42_non_home_discharge")),
        ("Non-home disch. (dual)", "m3care", HOSP_ABL.format(m="m3care", w="dualWindow", o="hospitalisation_non_home_discharge", r="m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_NHD_3_1hosp")),
        ("Non-home disch. (dual)", "fusemoe", HOSP_ABL.format(m="smp", w="dualWindow", o="hospitalisation_non_home_discharge", r="smp_smp_1e_1Lb_3_1hosp_non_home_discharge")),
        ("In-hosp. death (dual)", "concat", HOSP_ABL.format(m="concat", w="dualWindow", o="hospitalisation_in_hosp_death", r="concat_concat_lr1e5_wd1e4_s42_in_hosp_death")),
        ("In-hosp. death (dual)", "m3care", HOSP_ABL.format(m="m3care", w="dualWindow", o="hospitalisation_in_hosp_death", r="m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_IHD_3_1hosp")),
        ("In-hosp. death (dual)", "fusemoe", HOSP_ABL.format(m="smp", w="dualWindow", o="hospitalisation_in_hosp_death", r="smp_smp_1e_1Lb_3_1hosp_in_hosp_death")),
        ("Ext. stay > 7d (curr)", "concat", HOSP_ABL.format(m="concat", w="currWindow", o="hospitalisation_ext_stay_7", r="concat_concat_currWindow_ext_stay_7_3_1hosp_curr")),
        ("Ext. stay > 7d (curr)", "m3care", HOSP_ABL.format(m="m3care", w="currWindow", o="hospitalisation_ext_stay_7", r="m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_EXT_ST_currWin_3_1hosp")),
        ("Ext. stay > 7d (curr)", "fusemoe", HOSP_ABL.format(m="smp", w="currWindow", o="hospitalisation_ext_stay_7", r="smp_smp_currWindow_3_1hosp_rerun_02")),
        ("Non-home disch. (curr)", "concat", HOSP_ABL.format(m="concat", w="currWindow", o="hospitalisation_non_home_discharge", r="concat_concat_currWindow_non_home_discharge_3_1hosp_curr")),
        ("Non-home disch. (curr)", "m3care", HOSP_ABL.format(m="m3care", w="currWindow", o="hospitalisation_non_home_discharge", r="m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_NHD_currWin_3_1hosp")),
        ("Non-home disch. (curr)", "fusemoe", HOSP_ABL.format(m="smp", w="currWindow", o="hospitalisation_non_home_discharge", r="smp_smp_currWindow_non_home_discharge_3_1hosp_curr")),
        ("In-hosp. death (curr)", "concat", HOSP_ABL.format(m="concat", w="currWindow", o="hospitalisation_in_hosp_death", r="concat_concat_currWindow_in_hospital_death_3_1hosp_curr")),
        ("In-hosp. death (curr)", "m3care", HOSP_ABL.format(m="m3care", w="currWindow", o="hospitalisation_in_hosp_death", r="m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_IHD_currWin_3_1hosp")),
        ("In-hosp. death (curr)", "fusemoe", HOSP_ABL.format(m="smp", w="currWindow", o="hospitalisation_in_hosp_death", r="smp_smp_currWindow_in_hospital_death_3_1hosp_curr")),
    ],
}

FULL_TASK_NAME = {  # figures spell outcome names out in full
    "Mech. ventilation (mTAND)": "Mechanical ventilation (mTAND)",
    "Ext. stay > 7d (dual)": "Extended stay > 7 days (dual)",
    "Non-home disch. (dual)": "Non-home discharge (dual)",
    "In-hosp. death (dual)": "In-hospital death (dual)",
    "Ext. stay > 7d (curr)": "Extended stay > 7 days (curr)",
    "Non-home disch. (curr)": "Non-home discharge (curr)",
    "In-hosp. death (curr)": "In-hospital death (curr)",
}


def _cxr_available_share(run_dir):
    # mean CXR contribution within the subgroup that has a CXR; availability is inferred
    # from a non-zero per-sample CXR attribution (absent modalities attribute exactly 0)
    per = pd.read_csv(os.path.join(run_dir, "explanation", "mm_shap_per_sample.csv"))
    if "cxr_contribution" not in per:
        return None
    return 100 * per.loc[per["cxr_contribution"] > 0, "cxr_contribution"].mean()


mm_shap_pct = {
    pathway: [
        (outcome, method,
         json.load(open(os.path.join(run_dir, "explanation", "mm_shap_modality_contribution.json")))["global_contribution_pct"],
         pd.read_csv(os.path.join(run_dir, "metrics.csv"))["test_mcc"].iloc[-1],
         _cxr_available_share(run_dir) if pathway == "deterioration" else None)
        for outcome, method, run_dir in runs
    ]
    for pathway, runs in mm_shap_runs.items()
}

In [ ]:
# stacked-bar matrix: one horizontal 100% bar per run, grouped by outcome, one panel per
# pathway. Segment order is fixed (tabular/static first, then time series, ecg, cxr,
# notes); the prev-window variants are lighter tints of their curr-window hue.
MOD_STYLE = {  # modality -> (label, fill, inside-label ink)
    "tabular":    ("tabular",    "#6D2E8E", "white"),
    "static":     ("static",     "#6D2E8E", "white"),
    "ts":         ("ts (mTAND)", "#BE7C1C", "white"),
    "ts_prev":    ("ts prev",    "#E2BC7E", "#404040"),
    "ts_curr":    ("ts curr",    "#BE7C1C", "white"),
    "ecg":        ("ecg",        "#0E7C84", "white"),
    "ecg_curr":   ("ecg curr",   "#0E7C84", "white"),
    "cxr":        ("cxr",        "#B0413E", "white"),
    "cxr_prev":   ("cxr prev",   "#DBA5A3", "#404040"),
    "cxr_curr":   ("cxr curr",   "#B0413E", "white"),
    "notes_prev": ("notes prev", "#999999", "white"),
}
MOD_ORDER = {
    "deterioration": ["tabular", "ts", "ecg", "cxr"],
    "hospitalisation": ["static", "ts_prev", "ts_curr", "ecg_curr", "cxr_prev", "cxr_curr", "notes_prev"],
}
LEGEND_GRID = {  # legend columns as (top row, bottom row): static + curr on top, each prev under its curr
    "deterioration": [("tabular", None), ("ts", None), ("ecg", None), ("cxr", None)],
    "hospitalisation": [("static", None), ("ts_curr", "ts_prev"), ("ecg_curr", None),
                        ("cxr_curr", "cxr_prev"), (None, "notes_prev")],
}


def grid_legend(ax, pathway, present, handle):
    """Legend laid out as LEGEND_GRID, keeping only `present` modalities. Columns and rows
    left empty are dropped and remaining gaps padded with blank entries, since matplotlib
    fills legends column by column."""
    cols = [tuple(m if m in present else None for m in col) for col in LEGEND_GRID[pathway]]
    cols = [col for col in cols if any(col)]
    keep = [r for r in range(len(cols[0])) if any(col[r] for col in cols)]
    blank = plt.Line2D([], [], ls="", label="")
    ax.legend(handles=[handle(col[r]) if col[r] else blank for col in cols for r in keep],
              loc="upper center", bbox_to_anchor=(0.5, -0.075), ncol=len(cols),
              fontsize=7, frameon=False, handlelength=1.2, columnspacing=1.0)


fig, axes = plt.subplots(1, 2, figsize=(12.5, 7.2), dpi=600)
fig.patch.set_alpha(0.0)

for ax, pathway in zip(axes, mm_shap_pct):
    ax.patch.set_alpha(0.0)
    mods = MOD_ORDER[pathway]
    rows = mm_shap_pct[pathway]
    outcomes = list(dict.fromkeys(o for o, *_ in rows))

    ys, labels, best, y = [], [], [], 0
    for outcome in outcomes:
        ax.text(-0.01, y, FULL_TASK_NAME.get(outcome, outcome), transform=ax.get_yaxis_transform(),
                ha="right", va="center", fontsize=8, fontweight="bold", color="#404040")
        y -= 0.9
        best_mcc = max(mcc for o, _, _, mcc, _ in rows if o == outcome)
        for o, method, pct, mcc, cxr_avail in rows:
            if o != outcome:
                continue
            left = 0.0
            for mod in mods:
                v = pct.get(mod, 0.0)
                if v <= 0:
                    continue
                label, fill, ink = MOD_STYLE[mod]
                ax.barh(y, v, 0.62, left=left, color=fill,
                        edgecolor="white", linewidth=0.5)
                if v >= 9:  # direct-label the substantial segments only
                    ax.text(left + v / 2, y, f"{v:.0f}", ha="center", va="center",
                            fontsize=6.5, color=ink)
                left += v
            ax.text(101.5, y, f"{mcc:.3f}", ha="left", va="center", fontsize=6.5,
                    color="#404040", clip_on=False,
                    fontweight="bold" if mcc == best_mcc else "normal")
            ys.append(y)
            best.append(mcc == best_mcc)
            labels.append(f"* {method}" if mcc == best_mcc else method)
            y -= 0.9
        y -= 0.55  # gap between outcome groups

    ax.text(101.5, 0.35, "test MCC", ha="left", va="center", fontsize=6.5,
            color="#898781", style="italic", clip_on=False)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels, fontsize=8)
    for tick, is_best in zip(ax.get_yticklabels(), best):
        if is_best:
            tick.set_fontweight("bold")
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(0, 100)
    ax.set_ylim(y + 0.6, 0.6)
    ax.set_xlabel("MM-SHAP contribution (%)", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.set_title(pathway.capitalize(), fontsize=9)
    grid_legend(ax, pathway, mods,
                lambda m: Patch(facecolor=MOD_STYLE[m][1], edgecolor="white", label=MOD_STYLE[m][0]))

# fig.suptitle("MM-SHAP modality contribution: best run per fusion method, all outcomes", fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(os.path.join(OUT_DIR, "results_mm_shap_matrix.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
# conditional contribution dot-plot (sparse modalities only), styled like the matrix
# above: same grouped rows (bold outcome header, one row per fusion method) and the same
# modality colours; each dot is one sparse modality's mean MM-SHAP share WITHIN the
# subgroup of stays that has it. Because an absent modality attributes exactly 0, the
# missingness-diluted global share equals availability x the value shown.
SPARSE = {
    "deterioration": ["cxr"],
    "hospitalisation": ["ecg_curr", "cxr_curr", "ts_prev", "cxr_prev", "notes_prev"],
}

cond, availability = {}, {}
for pathway, runs in mm_shap_runs.items():
    for outcome, method, run_dir in runs:
        per = pd.read_csv(os.path.join(run_dir, "explanation", "mm_shap_per_sample.csv"))
        shares = {}
        for m in SPARSE[pathway]:
            c = f"{m}_contribution"
            if c not in per.columns:
                continue
            sc = per[c]
            shares[m] = 100 * sc[sc > 0].mean()
            availability[(pathway, m)] = (sc > 0).mean()
        cond[(outcome, method)] = shares

mm_shap_mcc = {(o, m): mcc for prows in mm_shap_pct.values() for o, m, _, mcc, _ in prows}
# x range from the data (next multiple of 10, at least 55) so no dot is clipped at the
# right edge; the test MCC column sits just beyond it
XMAX = max(55, 10 * np.ceil(max(v for sh in cond.values() for v in sh.values()) / 10))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 7.2), dpi=600)
fig.patch.set_alpha(0.0)

for ax, pathway in zip(axes, mm_shap_runs):
    ax.patch.set_alpha(0.0)
    rows = mm_shap_runs[pathway]
    outcomes = list(dict.fromkeys(o for o, _, _ in rows))

    ys, labels, best, y = [], [], [], 0
    for outcome in outcomes:
        ax.text(-0.01, y, FULL_TASK_NAME.get(outcome, outcome), transform=ax.get_yaxis_transform(),
                ha="right", va="center", fontsize=8, fontweight="bold", color="#404040")
        y -= 0.9
        best_mcc = max(mm_shap_mcc[o, m] for o, m, _ in rows if o == outcome)
        for o, method, _ in rows:
            if o != outcome:
                continue
            mcc = mm_shap_mcc[o, method]
            # nudge near-coincident dots apart vertically so none hides another
            items = sorted(cond[(o, method)].items(), key=lambda kv: kv[1])
            prev_val, toggle = None, 1
            for m, share in items:
                off = 0.0
                if prev_val is not None and share - prev_val < 1.5:
                    off, toggle = 0.22 * toggle, -toggle
                prev_val = share
                ax.scatter([share], [y + off], s=32, color=MOD_STYLE[m][1],
                           edgecolor="white", linewidth=0.5, zorder=2)
            ax.text(XMAX + 1.5, y, f"{mcc:.3f}", ha="left", va="center", fontsize=6.5,
                    color="#404040", clip_on=False,
                    fontweight="bold" if mcc == best_mcc else "normal")
            ys.append(y)
            best.append(mcc == best_mcc)
            labels.append(f"* {method}" if mcc == best_mcc else method)
            y -= 0.9
        y -= 0.55  # gap between outcome groups

    ax.text(XMAX + 1.5, 0.35, "test MCC", ha="left", va="center", fontsize=6.5,
            color="#898781", style="italic", clip_on=False)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels, fontsize=8)
    for tick, is_best in zip(ax.get_yticklabels(), best):
        if is_best:
            tick.set_fontweight("bold")
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(0, XMAX)
    ax.set_ylim(y + 0.6, 0.6)
    ax.set_xlabel("MM-SHAP contribution when the modality is present (%)", fontsize=8)
    ax.tick_params(axis="x", labelsize=7)
    ax.grid(axis="x", color="#E3E3E3", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.set_title(pathway.capitalize(), fontsize=9)
    mods = [m for m in SPARSE[pathway] if (pathway, m) in availability]
    grid_legend(ax, pathway, mods,
                lambda m: plt.Line2D([], [], marker="o", ls="", color=MOD_STYLE[m][1],
                                     label=f"{MOD_STYLE[m][0]} ({availability[(pathway, m)]:.0%} of stays)"))

# fig.suptitle("MM-SHAP contribution of sparse modalities when present", fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(os.path.join(OUT_DIR, "results_mm_shap_conditional.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

# Fairness headline: equal opportunity

Equal opportunity ratio (worst-group / best-group sensitivity) per sensitive attribute, one dot per best run across both pathways. Sex sits near parity, while age, ethnicity, insurance and CXR availability sit well below parity — and the three fusion methods intermix within every row, so the gaps are data-driven, not architectural. Runs where a subgroup has zero true positives (the rare outcomes cardiac arrest and One-day mortality on Age/Ethnicity) yield an undefined ratio, floored to a 0.001 sentinel by src/evaluation/fairness.py, and are omitted from the plot. Uses `mm_shap_runs` from the data cell above.

In [ ]:
# fairness headline figure: equal opportunity ratio (worst group / best group sensitivity)
# per sensitive attribute, one dot per best run. Marker shape = fusion method, colour =
# outcome, outcome, dark tick = median across runs.
FAIR_ATTRS = {
    "deterioration": ["Sex", "Age", "Ethnicity", "CXR Availability"],
    "hospitalisation": ["Sex", "Age", "Ethnicity", "Insurance"],
}
FAIR_MARK = {"concat": "o", "m3care": "s", "fusemoe": "^",
             "m3care + mTAND": "s", "fusemoe + mTAND": "^"}
FAIR_LANE = {"concat": 0.25, "m3care": 0.0, "fusemoe": -0.25,
             "m3care + mTAND": 0.0, "fusemoe + mTAND": -0.25}
FAIR_OUTCOME_NAME = {  # legend uses full outcome names, windows spelled out
    "Ext. stay > 7d (dual)": "Extended stay > 7 days (dual window)",
    "Non-home disch. (dual)": "Non-home discharge (dual window)",
    "In-hosp. death (dual)": "In-hospital death (dual window)",
    "Ext. stay > 7d (curr)": "Extended stay > 7 days (current window)",
    "Non-home disch. (curr)": "Non-home discharge (current window)",
    "In-hosp. death (curr)": "In-hospital death (current window)",
    "Mech. ventilation (mTAND)": "Mechanical ventilation (mTAND)",
}
OUTCOME_COLOURS = ["#6D2E8E", "#0E7C84", "#BE7C1C", "#3B4C61", "#B0413E", "#999999"]

eop = {}  # (pathway, attribute) -> list of (outcome, method, value)
for pathway, runs in mm_shap_runs.items():
    for outcome, method, run_dir in runs:
        f = pd.read_csv(os.path.join(run_dir, "fairness", "fairness_summary.csv"))
        for _, r in f[f["metric"] == "EOP"].iterrows():
            # values floored to the 0.001 sentinel are undefined (a subgroup with zero
            # true positives), not measured disparities - omit them from the plot
            if r["attribute"] in FAIR_ATTRS[pathway] and r["value"] > 0.001:
                eop.setdefault((pathway, r["attribute"]), []).append((outcome, method, r["value"]))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.2), dpi=600)
fig.patch.set_alpha(0.0)

for ax, pathway in zip(axes, mm_shap_runs):
    ax.patch.set_alpha(0.0)
    outcomes = list(dict.fromkeys(o for o, _, _ in mm_shap_runs[pathway]))
    colour = dict(zip(outcomes, OUTCOME_COLOURS))
    attrs = FAIR_ATTRS[pathway]
    for i, attr in enumerate(attrs):
        y = -i
        vals = eop[(pathway, attr)]
        for outcome, method, v in vals:
            ax.scatter([v], [y + FAIR_LANE[method]], s=30, marker=FAIR_MARK[method],
                       color=colour[outcome], edgecolor="white", linewidth=0.4, zorder=2)
        ax.scatter([pd.Series([v for *_, v in vals]).median()], [y], s=220, marker="|",
                   color="#404040", zorder=3)
    ax.axvline(1.0, color="#C3C2B7", lw=0.9, zorder=1)
    ax.set_yticks([-i for i in range(len(attrs))])
    ax.set_yticklabels(attrs, fontsize=8)
    ax.tick_params(axis="y", length=0)
    ax.set_ylim(-len(attrs) + 0.5, 0.62)
    ax.set_xlim(0, 1.04)
    ax.tick_params(axis="x", labelsize=7)
    ax.set_xlabel("equal opportunity ratio (worst-group / best-group sensitivity)", fontsize=8)
    ax.grid(axis="x", color="#E3E3E3", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.set_title(pathway.capitalize(), fontsize=9)
    ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=colour[o],
                                  label=FAIR_OUTCOME_NAME.get(o, o))
                       for o in outcomes],
              loc="upper center", bbox_to_anchor=(0.5, -0.11),
              ncol=2, fontsize=7, frameon=False, handlelength=1.2, columnspacing=1.0)

fig.legend(handles=[plt.Line2D([], [], marker=mk, ls="", color="#52514e", label=lb)
                    for lb, mk in (("concat", "o"), ("m3care", "s"), ("fusemoe", "^"))],
           loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=7.5, frameon=False,
           title="marker = fusion method, | = median across runs",
           title_fontsize=7)
fig.suptitle("Fairness of the best runs: equal opportunity across sensitive attributes",
             fontsize=14)
plt.tight_layout(rect=(0, 0.06, 1, 0.95))
fig.savefig(os.path.join(OUT_DIR, "results_fairness_eop.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
# compact companion for the fairness slide: selection (alert) rate by CXR availability for
# the fusemoe mechanical-ventilation best run - the informative-missingness point in one
# small panel (patients with a CXR are alerted on 4x as often)
_fair_json = os.path.join(
    "/data/logs/ablation/smp/deterioration/allStay/averageRecord/allCohort/normalised_ohe/"
    "deterioration_mechanical_ventilation/runs/smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe",
    "fairness", "test_fairness_metrics.json")
with open(_fair_json) as fh:
    _groups = json.load(fh)["attributes"]["CXR Availability"]["groups"]

fig, ax = plt.subplots(figsize=(3.1, 1.5), dpi=600)
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)
order = ["CXR absent", "CXR present"]
for i, gname in enumerate(order):
    g = _groups[gname]
    ax.bar(i, 100 * g["selection_rate"], 0.62, color="#B0413E",
           edgecolor="white", linewidth=0.5)
    ax.text(i, 100 * g["selection_rate"] + 0.6, f"{100 * g['selection_rate']:.0f}%",
            ha="center", va="bottom", fontsize=8, color="#404040")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{g}\n(n={_groups[g]['count']:,})" for g in order], fontsize=8)
ax.set_ylabel("selection rate (%)", fontsize=8)
ax.set_ylim(0, 21)
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", length=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_title("Selection rate by CXR availability\n(mechanical ventilation, fusemoe)", fontsize=8.5)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "results_fairness_cxr_selection.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
# compact companion for the fairness slide: selection (alert) rate by insurance for the
# concat extended-stay best run (dual window) - Medicare patients are alerted on at 2.6x
# the private-insurance rate, mirroring the age gradient
_fair_json = os.path.join(
    "/data/logs/ablation/concat/hospitalisation/dualWindow/averageRecord/optionalHistory/"
    "normalised_ohe/hospitalisation_ext_stay_7/runs/concat_concat_lr1e5_wd1e4_s42",
    "fairness", "test_fairness_metrics.json")
with open(_fair_json) as fh:
    _groups = json.load(fh)["attributes"]["Insurance"]["groups"]

fig, ax = plt.subplots(figsize=(3.1, 1.5), dpi=600)
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)
order = ["Medicaid", "Medicare", "Other", "Private"]
for i, gname in enumerate(order):
    g = _groups[gname]
    ax.bar(i, 100 * g["selection_rate"], 0.62, color="#BE7C1C",
           edgecolor="white", linewidth=0.5)
    ax.text(i, 100 * g["selection_rate"] + 1.2, f"{100 * g['selection_rate']:.0f}%",
            ha="center", va="bottom", fontsize=8, color="#404040")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{g}\n(n={_groups[g]['count']:,})" for g in order], fontsize=7.5)
ax.set_ylabel("selection rate (%)", fontsize=8)
ax.set_ylim(0, 46)
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", length=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_title("Selection rate by insurance\n(extended stay > 7 days, concat, dual window)", fontsize=8.5)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "results_fairness_insurance_selection.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

# Figure 5.3: joint routing in the best runs

Routing behaviour of the joint router in the best fusemoe (`smp`) run of each task, on the test set. Three measures per run, read from `<run>/expert/test_expert_gate_weights.csv`:
(a) share of top-2 selections taken by the least-used of the seven routed experts (even use = 1/7),
(b) mean routing weight of the higher-weighted selected expert (equal split = 0.5),
(c) Cramér's V between the top expert and the modality-availability pattern.
Deterioration has two availability patterns, hospitalisation eight (current window) or 30 (dual window), so (c) is not directly comparable across pathways. Uses `mm_shap_runs` from the MM-SHAP data cell above.

In [ ]:
# Figure 5.3: routing summary of the joint router, one row per best fusemoe run (test set).
# Rows follow the report order: deterioration outcomes, then hospitalisation dual and current window.
from scipy.stats import chi2_contingency

ROUTING_ROWS = {  # mm_shap_runs outcome label -> row label in the figure
    "deterioration": {
        "Mechanical ventilation": "Mechanical ventilation",
        "Cardiac arrest": "Cardiac arrest",
        "ICU within 24h": "ICU within 24 h",
        "ICU admission": "ICU admission",
        "One-day mortality": "One-day mortality",
    },
    "hospitalisation": {
        "Ext. stay > 7d (dual)": "Extended stay > 7 days (dual)",
        "Non-home disch. (dual)": "Non-home discharge (dual)",
        "In-hosp. death (dual)": "In-hospital death (dual)",
        "Ext. stay > 7d (curr)": "Extended stay > 7 days (current)",
        "Non-home disch. (curr)": "Non-home discharge (current)",
        "In-hosp. death (curr)": "In-hospital death (current)",
    },
}


def routing_measures(run_dir):
    """Least-used expert share, mean top routing weight and Cramér's V (top expert x availability)."""
    g = pd.read_csv(os.path.join(run_dir, "expert", "test_expert_gate_weights.csv"))
    w = g.filter(like="_gate_weight").to_numpy()
    load = (w > 0).sum(axis=0)  # top-2 selections per expert
    pattern = g[[c for c in g.columns if c.startswith("has_")]].astype(str).agg("".join, axis=1)
    table = pd.crosstab(pattern, g["top_expert"])
    chi2 = chi2_contingency(table, correction=False)[0]
    return {
        "least_used": load.min() / load.sum(),
        "top_weight": np.sort(w, axis=1)[:, -1].mean(),
        "cramers_v": np.sqrt(chi2 / (len(g) * (min(table.shape) - 1))),
        "patterns": pattern.nunique(),
        "n": len(g),
    }


fusemoe_runs = {(p, o): d for p, runs in mm_shap_runs.items() for o, m, d in runs if m == "fusemoe"}
routing = pd.DataFrame([
    {"pathway": p, "label": label, **routing_measures(fusemoe_runs[p, o])}
    for p, rows in ROUTING_ROWS.items() for o, label in rows.items()
])
routing.to_csv(os.path.join(OUT_DIR, "results_routing_summary.csv"), index=False)
display(routing.round(3))

ROUTING_COLOURS = {"deterioration": "#6D2E8E", "hospitalisation": "#0E7C84"}

y = -np.arange(len(routing), dtype=float)
sep = (y[routing.pathway.eq("deterioration")].min() + y[routing.pathway.eq("hospitalisation")].max()) / 2

fig, axes = plt.subplots(1, 3, figsize=(8.5, 4.2), sharey=True)
panels = [  # column, x label, x limits, reference line, scale, value format
    ("least_used", "Least-used expert share (%)", (0, 16), 100 / 7, 100, "{:.1f}"),
    ("top_weight", "Mean top routing weight", (0.45, 0.65), 0.5, 1, "{:.2f}"),
    ("cramers_v", "Cramér's V", (0, 1.05), None, 1, "{:.2f}"),
]
for ax, (col, xlabel, xlim, ref, scale, fmt) in zip(axes, panels):
    for p in ROUTING_COLOURS:
        m = routing.pathway.eq(p)
        ax.plot(routing.loc[m, col] * scale, y[m], "o", color=ROUTING_COLOURS[p],
                ms=6, label=p.capitalize())
        for val, yy in zip(routing.loc[m, col] * scale, y[m]):
            ax.annotate(fmt.format(val), (val, yy), xytext=(6, 0), textcoords="offset points",
                        va="center", fontsize=8)
    if ref is not None:
        ax.axvline(ref, color="grey", lw=0.8, ls="--")
    ax.axhline(sep, color="grey", lw=0.6)
    ax.set_xlim(*xlim)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", color="0.9", lw=0.6)
    ax.set_axisbelow(True)
axes[0].set_xticks([0, 5, 10, 15])
axes[0].set_yticks(y)
axes[0].set_yticklabels(routing.label)
axes[0].set_ylim(y.min() - 0.6, y.max() + 0.6)
handles, labels = axes[0].get_legend_handles_labels()
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "results_routing_summary.png"), dpi=600, transparent=True, facecolor="white")
plt.show()